# Trajectory and velocity analysis of scRNAseq COLON data 

This notebook will guide your through the analysis... This is an example for the **HEALTHY** dataset.

## 0. Imports and settings

In [ ]:
import scanpy as sc
import scvelo as scv
import numpy as np
import pandas as pd
import mnnpy
import matplotlib.pyplot as plt
from matplotlib import colors
import seaborn as sb
import scanpy.api as scapi
import scrublet as scr
import doubletdetection

# Set some decent font (probably will have to change to Helvetica for publication)
from matplotlib import rcParams
rcParams['font.family'] = ['Georgia serif']

# Box for text background in PAGA
bbox = dict(bbox=dict(boxstyle="round", ec='white', fc="white", alpha=0.5, linewidth=0))

# List of genes associated with cell cycle.
cell_cycle_genes = [x.strip() for x in open('data/cell_cycle_genes.txt')]
cell_cycle_genes = cell_cycle_genes[1:]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

# Proliferation markers
ProlifMarkers = ['RAD51C','SLC7A2','CCDC18','DCTD','RAD54B','BARD1','KLHL23','FIGNL1','POLA1','DEPDC1','PPIL5','PPAT','C6ORF150','XRCC2','SKA1','SLC12A2','RCC2','KIF18A','KIF11','ESCO2','E2F7','RAD54L','CHEK1','PRKD3','NA','CLSPN','KIAA0101','DTL','SLC7A2','ATAD5','POLE','FANCB','CENPA','NEXN','PPIL5','FAM111A','TTPA','CDC7','NAP1L1','HEMGN','KNTC1','PRKD3','TBC1D19','SKA3','NCAPG2','POLE2','EXO1','CENPI','SGOL2','CENPN','DTL','CENPN','NEIL3','EXO1','HMMR','RAD54L','BUB1','MCM3','NRM','CNN3','ALMS1','TIMELESS','ATAD2','RRP1B','AURKB','SLC16A12','RAD51C','MCM3','TPX2','C1ORF135','KIFC1','TBC1D4','CHEK1','ZWILCH','SCML2','FADS1','GINS2','MYC','TIMELESS','AQP4','PCK2','CDC45L','ANKRD26','PPAT','HELLS','C15ORF42','ZNF473','CDCA2','NA','HMMR','UBE2T','BRCA1','KIF11','KIF20B','PLK4','PRR11','TMEFF1','CNN3','NA','MCM5','MCM2','GINS1','TMEM107','ERCC6L','CASC5','IFITM3','RAD51AP1','RAD51AP1','BUB1','NCAPH','CBX6','DTL','BLM','GINS2','PRR11','9430037O13RIK','MORC4','PFAS','FGR','RAD54B','C6ORF173','NCAPH','NA','PALB2','MELK','NSL1','CDCA2','NAP1L1','C6ORF150','MBD4','MCM3','CENPF','MCM7','RACGAP1','TIPIN','MYB','C14ORF106','C16ORF48','RAD18','CENPE','RAD51AP1','CHEK1','MCM3','UHRF1','CEP55','FOXM1','SLFN13','IQGAP3','TUBE1','NUF2','CCDC99','RAD51','FAM72A','CDCA5','SMC2','KIF18B','MCM3','RACGAP1','GLS2','DUT','TMEM107','BRCA1','AU016916','MYB','2700099C18RIK','GTSE1','RAD54L','HELLS','WDR35','PLK1','TIMP3','RPA2','CEP55','NA','RAD18','C6ORF167','C4ORF21','PRIM2','C12ORF48','BUB1B','C1ORF112','NA','LAMC1','ANLN','TRIM37','RRM2','CCNA2','HAUS5','ESPL1','TCF19','FOXM1','DDX11','KIF2C','CCNE1','PUS7','LPHN1','KBTBD6','NA','TXNDC16','DACH1','MASTL','TRIM37','ZRANB3','SIVA1','2810454L23RIK','CELSR2','PSIP1','MCM8','SHCBP1','ASF1B','KCNN4','AU020206','PRIM1','CKS1B','CHAF1B','MCM2','RCC2','RASA3','DUT','CTPS','BAG2','SIVA1','CKLF','PRIM2','DUS4L','TMEM194B','TUBB','HAUS6','DCAF12L1','MARCKS','SETDB1','2810408B13RIK','SYCE2','UNG','NA','SEMA3C','CLCA4','MLF1IP','PAICS','MCM7','POLD1','C17ORF53','E2F1','ZC3H7B','TRAIP','LOC388559','MPHOSPH9','WDHD1','BCL7A','NA','ATIC','CCBE1','TSGA14','HELLS','MASTL','SPAG5','LYSMD2','C15ORF23','CEP76','DNAJC18','TAF4B','CDC6','SMC2','CENPE','4930513N10RIK','LYAR','CDK2','FOXM1','CKLF','NCAPD2','SLFN13','RPUSD2','VRK1','CYP39A1','DLGAP5','INCENP','PBK','TMEFF1','NUSAP1','NEK2','C3ORF26','VRK1','PSMC3IP','POLI','NFATC2','D17H6S56E-5','PGM2L1','AURKB','CEP97','CKAP2L','FANCB','DNMT1','CCNF','BRCA1','ASPM','HMMR','HMGA2','DSN1','NA','CDCA5','PGM2L1','IGF1R','ZNF367','WDR34','NUP210','EXOSC8','GEN1','LIG1','TPX2','HMMR','TK1','PLK4','TRIM68','HAUS6','FAM54A','C8ORF79','AEN','ZNRF3','PSMC3IP','CCDC14','DNA2','CENPP','C21ORF91','DLGAP5','DCK','SLC7A5','HAUS4','CHTF18','SEMA4D','PGM2L1','ZBTB25','UHRF1','TSGA14','BCKDHB','MCM4','KIF22','CHEK1','NAP1L1','RRM1','FTSJD1','SGOL1','MDN1','FEN1','RRM2','CKLF','NEK2','QSOX2','RRP15','NCAPG','AURKA','MYBL2','NA','PAICS','PIP4K2B','SPC24','DNA2','PRC1','RPP40','KIF24','MPHOSPH9','TMEM173','TMEM48','NIN','WDR76','ZNF275','UTP15','3110040M04RIK','MTHFD2','TACC3','NA','RBBP6','CENPF','ZMYND19','SKA2','MTBP','NDE1','MYBL2','EXOSC2','DIAPH3','CDC2','CP110','KIAA0649','GEMIN5','DUSP7','POLR1E','NUP133','TRIM37','TSPAN12','NAP1L1','PRC1','GEMIN4','GINS3','TOPBP1','NEUROG3','MPHOSPH9','CASP12','WWTR1','NAP1L1','MCM7','RFC3','ZMYND19','KIF4A','LMNB1','RFC3','BBS7','ILF3','ZNF704','GSTCD','PRIM1','NA','ZBTB16','NA','DKC1','QTRT1','TRIP13','SLC1A5','CDCA7L','C14ORF143','MCM5','TRIP13','2810442I21RIK','DNAH11','4933439C10RIK','POLD1','CKLF','CEP192','MIRHG1','ZFP783','NUP85','C3ORF26','TLR2','C6ORF167']
ProlifMarkers = list(set(ProlifMarkers))

## 1. Load data

All files are in the `.loom` format. They have been preprcoessed with *cellranger* (demultiplexing, alignment and counting) and *velocyto* (annotation of spliced/unspliced counts). 

After loading in the file, we have to remove some of the unnecssary data fields generated by previous processing. What we want in the end is just the matrix of raw counts, and matrices of spliced and unspliced reads.

In [ ]:
sample_name = "healthy0"
file = "data/{}.loom".format(sample_name)
adata0 = scv.read_loom(file, sparse=True, cleanup=True)
adata0.var_names_make_unique()
# Delete unnecessary data
del adata0.obs['Clusters']
del adata0.obs['_X']
del adata0.obs['_Y']
del adata0.var['Accession']
del adata0.var['Chromosome']
del adata0.var['End']
del adata0.var['Start']
del adata0.var['Strand']
del adata0.layers['ambiguous']

sample_name = "healthy1"
file = "data/{}.loom".format(sample_name)
adata1 = scv.read_loom(file, sparse=True, cleanup=True)
adata1.var_names_make_unique()
del adata1.obs['Clusters']
del adata1.obs['_X']
del adata1.obs['_Y']
del adata1.var['Accession']
del adata1.var['Chromosome']
del adata1.var['End']
del adata1.var['Start']
del adata1.var['Strand']
del adata1.layers['ambiguous']

sample_name = "healthy7"
file = "data/{}.loom".format(sample_name)
adata7 = scv.read_loom(file, sparse=True, cleanup=True)
adata7.var_names_make_unique()
del adata7.obs['Clusters']
del adata7.obs['_X']
del adata7.obs['_Y']
del adata7.var['Accession']
del adata7.var['Chromosome']
del adata7.var['End']
del adata7.var['Start']
del adata7.var['Strand']
del adata7.layers['ambiguous']

sample_name = "healthy8"
file = "data/{}.loom".format(sample_name)
adata8 = scv.read_loom(file, sparse=True, cleanup=True)
adata8.var_names_make_unique()
del adata8.obs['Clusters']
del adata8.obs['_X']
del adata8.obs['_Y']
del adata8.var['Accession']
del adata8.var['Chromosome']
del adata8.var['End']
del adata8.var['Start']
del adata8.var['Strand']
del adata8.layers['ambiguous']

sample_name = "inflamed3"
file = "data/{}.loom".format(sample_name)
adata3 = scv.read_loom(file, sparse=True, cleanup=True)
adata3.var_names_make_unique()
# Delete unnecessary data
del adata3.obs['Clusters']
del adata3.obs['_X']
del adata3.obs['_Y']
del adata3.var['Accession']
del adata3.var['Chromosome']
del adata3.var['End']
del adata3.var['Start']
del adata3.var['Strand']
del adata3.layers['ambiguous']

sample_name = "inflamed4"
file = "data/{}.loom".format(sample_name)
adata4 = scv.read_loom(file, sparse=True, cleanup=True)
adata4.var_names_make_unique()
# Delete unnecessary data
del adata4.obs['Clusters']
del adata4.obs['_X']
del adata4.obs['_Y']
del adata4.var['Accession']
del adata4.var['Chromosome']
del adata4.var['End']
del adata4.var['Start']
del adata4.var['Strand']
del adata4.layers['ambiguous']

sample_name = "inflamed6"
file = "data/{}.loom".format(sample_name)
adata6 = scv.read_loom(file, sparse=True, cleanup=True)
adata6.var_names_make_unique()
# Delete unnecessary data
del adata6.obs['Clusters']
del adata6.obs['_X']
del adata6.obs['_Y']
del adata6.var['Accession']
del adata6.var['Chromosome']
del adata6.var['End']
del adata6.var['Start']
del adata6.var['Strand']
del adata6.layers['ambiguous']

sample_name = "inflamed10"
file = "data/{}.loom".format(sample_name)
adata10 = scv.read_loom(file, sparse=True, cleanup=True)
adata10.var_names_make_unique()
# Delete unnecessary data
del adata10.obs['Clusters']
del adata10.obs['_X']
del adata10.obs['_Y']
del adata10.var['Accession']
del adata10.var['Chromosome']
del adata10.var['End']
del adata10.var['Start']
del adata10.var['Strand']
del adata10.layers['ambiguous']

sample_name = "noninflamed2"
file = "data/{}.loom".format(sample_name)
adata2 = scv.read_loom(file, sparse=True, cleanup=True)
adata2.var_names_make_unique()
# Delete unnecessary data
del adata2.obs['Clusters']
del adata2.obs['_X']
del adata2.obs['_Y']
del adata2.var['Accession']
del adata2.var['Chromosome']
del adata2.var['End']
del adata2.var['Start']
del adata2.var['Strand']
del adata2.layers['ambiguous']

sample_name = "noninflamed5"
file = "data/{}.loom".format(sample_name)
adata5 = scv.read_loom(file, sparse=True, cleanup=True)
adata5.var_names_make_unique()
# Delete unnecessary data
del adata5.obs['Clusters']
del adata5.obs['_X']
del adata5.obs['_Y']
del adata5.var['Accession']
del adata5.var['Chromosome']
del adata5.var['End']
del adata5.var['Start']
del adata5.var['Strand']
del adata5.layers['ambiguous']

sample_name = "noninflamed9"
file = "data/{}.loom".format(sample_name)
adata9 = scv.read_loom(file, sparse=True, cleanup=True)
adata9.var_names_make_unique()
# Delete unnecessary data
del adata9.obs['Clusters']
del adata9.obs['_X']
del adata9.obs['_Y']
del adata9.var['Accession']
del adata9.var['Chromosome']
del adata9.var['End']
del adata9.var['Start']
del adata9.var['Strand']
del adata9.layers['ambiguous']

## 2. Quality control

Before we can analyse the samples properly we need to filter the cells and genes based on their quality. We start by joining the samples together and calculating some standard quality metrics. 

In [ ]:
# Concatenate the files to calculate joint QC metrics
adata_qc = adata0.concatenate(adata1, adata7, adata8, adata3, adata4, adata6, adata10, adata2, adata5, adata9,
                              batch_key='Sample', batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8',
                                                                    'inflamed3', 'inflamed4', 'inflamed6', 'inflamed10',
                                                                    'noninflamed2', 'noninflamed5', 'noninflamed9'])

# Calculate QC covariates
adata_qc.obs['n_counts'] = adata_qc.X.sum(1)
adata_qc.obs['n_spliced'] = adata_qc.layers['spliced'].sum(1)
adata_qc.obs['n_unspliced'] = adata_qc.layers['unspliced'].sum(1)
adata_qc.obs['n_genes'] = (adata_qc.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata_qc.var_names.str.startswith('MT-')
adata_qc.obs['percent_mito'] = np.sum(
    adata_qc[:, mito_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)
ribo_genes = adata_qc.var_names.str.startswith('RP')
adata_qc.obs['percent_ribo'] = np.sum(
    adata_qc[:, ribo_genes].X, axis=1) / np.sum(adata_qc.X, axis=1)

# Initial plotting settings for more detailed scatter and dist plots.
scv.settings.set_figure_params('scvelo', dpi=150, vector_friendly=False)

sc.pl.violin(adata_qc, ['n_genes'], jitter=0.4, groupby='Sample', rotation=45)
sc.pl.violin(adata_qc, ['n_counts'], jitter=0.4, groupby='Sample', rotation=45)
sc.pl.violin(adata_qc, ['percent_mito'], jitter=0.4, groupby='Sample', rotation=45)
sc.pl.violin(adata_qc, ['percent_ribo'], jitter=0.4, groupby='Sample', rotation=45)
sc.pl.violin(adata_qc, ['n_spliced'], jitter=0.4, groupby='Sample', rotation=45)
sc.pl.violin(adata_qc, ['n_unspliced'], jitter=0.4, groupby='Sample', rotation=45)

As can be seen in the figure above, there are significant differences in the distributions of quality metrics in the four samples. This is why filtering based on those metrics is done separately for each of the samples.

### 2.1 healthy0 QC 

In [ ]:
# Calculate QC covariates
adata0.obs['n_counts'] = adata0.X.sum(1)
adata0.obs['log_counts'] = np.log(adata0.obs['n_counts'])
adata0.obs['n_spliced'] = adata0.layers['spliced'].sum(1)
adata0.obs['n_unspliced'] = adata0.layers['unspliced'].sum(1)
adata0.obs['n_genes'] = (adata0.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata0.var_names.str.startswith('MT-')
adata0.obs['percent_mito'] = np.sum(
    adata0[:, mito_genes].X, axis=1) / np.sum(adata0.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata0.var_names.str.startswith('RP')
adata0.obs['percent_ribo'] = np.sum(
    adata0[:, ribo_genes].X, axis=1) / np.sum(adata0.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata0.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata0.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata0.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata0.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata0, min_counts=1250)
sc.pp.filter_cells(adata0, max_counts=18750)
adata0 = adata0[adata0.obs['n_unspliced']>600]
sc.pp.filter_cells(adata0, min_genes=250)

In [ ]:
adata0 = adata0[adata0.obs['percent_mito'] < 0.3, :]

In [ ]:
adata0 = adata0[adata0.obs['doublet'] != '1']
adata0 = adata0[adata0.obs['doublet_score'] < 0.3]

### 2.2 healthy1 QC

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata1.obs['n_counts'] = adata1.X.sum(1)
adata1.obs['log_counts'] = np.log(adata1.obs['n_counts'])
adata1.obs['n_genes'] = (adata1.X > 0).sum(1)
adata1.obs['n_spliced'] = adata1.layers['spliced'].sum(1)
adata1.obs['n_unspliced'] = adata1.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata1.var_names.str.startswith('MT-')
adata1.obs['percent_mito'] = np.sum(
    adata1[:, mito_genes].X, axis=1) / np.sum(adata1.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata1.var_names.str.startswith('RP')
adata1.obs['percent_ribo'] = np.sum(
    adata1[:, ribo_genes].X, axis=1) / np.sum(adata1.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata1.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata1.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata1.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata1.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata1, min_counts=2100)
sc.pp.filter_cells(adata1, max_counts=26000)
adata1 = adata1[adata1.obs['n_unspliced']>700]
sc.pp.filter_cells(adata1, min_genes=900)

In [ ]:
adata1 = adata1[adata1.obs['percent_mito'] < 0.3, :]

In [ ]:
adata1 = adata1[adata1.obs['doublet'] != '1']
adata1 = adata1[adata1.obs['doublet_score'] < 0.3]

### 2.3 healthy7 QC

In [ ]:
# Calculate QC covariates
adata7.obs['n_counts'] = adata7.X.sum(1)
adata7.obs['log_counts'] = np.log(adata7.obs['n_counts'])
adata7.obs['n_genes'] = (adata7.X > 0).sum(1)
adata7.obs['n_spliced'] = adata7.layers['spliced'].sum(1)
adata7.obs['n_unspliced'] = adata7.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata7.var_names.str.startswith('MT-')
adata7.obs['percent_mito'] = np.sum(
    adata7[:, mito_genes].X, axis=1) / np.sum(adata7.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata7.var_names.str.startswith('RP')
adata7.obs['percent_ribo'] = np.sum(
    adata7[:, ribo_genes].X, axis=1) / np.sum(adata7.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata7.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata7.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata7.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata7.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata7, min_counts=1600)
sc.pp.filter_cells(adata7, max_counts=18000)
adata7 = adata7[adata7.obs['n_unspliced']>750]
sc.pp.filter_cells(adata7, min_genes=500)

In [ ]:
adata7 = adata7[adata7.obs['percent_mito'] < 0.3, :]

In [ ]:
adata7 = adata7[adata7.obs['doublet'] != '1']
adata7 = adata7[adata7.obs['doublet_score'] < 0.3]

### 2.4 healthy8 QC

The analysis and its reasoning is analogous to the previous one. See section **2.1** for describtion.

In [ ]:
# Calculate QC covariates
adata8.obs['n_counts'] = adata8.X.sum(1)
adata8.obs['log_counts'] = np.log(adata8.obs['n_counts'])
adata8.obs['n_genes'] = (adata8.X > 0).sum(1)
adata8.obs['n_spliced'] = adata8.layers['spliced'].sum(1)
adata8.obs['n_unspliced'] = adata8.layers['unspliced'].sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata8.var_names.str.startswith('MT-')
adata8.obs['percent_mito'] = np.sum(
    adata8[:, mito_genes].X, axis=1) / np.sum(adata8.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata8.var_names.str.startswith('RP')
adata8.obs['percent_ribo'] = np.sum(
    adata8[:, ribo_genes].X, axis=1) / np.sum(adata8.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata8.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata8.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata8.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata8.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata8, min_counts=1600)
sc.pp.filter_cells(adata8, max_counts=16000)
adata8 = adata8[adata8.obs['n_unspliced']>800]
sc.pp.filter_cells(adata8, min_genes=500)

In [ ]:
adata8 = adata8[adata8.obs['percent_mito'] < 0.3, :]

In [ ]:
adata8 = adata8[adata8.obs['doublet'] != '1']
adata8 = adata8[adata8.obs['doublet_score'] < 0.3]

### 2.5 inflamed3 QC 

In [ ]:
# Calculate QC covariates
adata3.obs['n_counts'] = adata3.X.sum(1)
adata3.obs['log_counts'] = np.log(adata3.obs['n_counts'])
adata3.obs['n_spliced'] = adata3.layers['spliced'].sum(1)
adata3.obs['n_unspliced'] = adata3.layers['unspliced'].sum(1)
adata3.obs['n_genes'] = (adata3.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata3.var_names.str.startswith('MT-')
adata3.obs['percent_mito'] = np.sum(
    adata3[:, mito_genes].X, axis=1) / np.sum(adata3.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata3.var_names.str.startswith('RP')
adata3.obs['percent_ribo'] = np.sum(
    adata3[:, ribo_genes].X, axis=1) / np.sum(adata3.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata3.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata3.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata3.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata3.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata3, min_counts=2000)
sc.pp.filter_cells(adata3, max_counts=23500)
adata3 = adata3[adata3.obs['n_unspliced']>750]
sc.pp.filter_cells(adata3, min_genes=750)

In [ ]:
adata3 = adata3[adata3.obs['percent_mito'] < 0.3, :]

In [ ]:
adata3 = adata3[adata3.obs['doublet'] != '1']
adata3 = adata3[adata3.obs['doublet_score'] < 0.3]

### 2.6 inflamed4 QC

In [ ]:
# Calculate QC covariates
adata4.obs['n_counts'] = adata4.X.sum(1)
adata4.obs['log_counts'] = np.log(adata4.obs['n_counts'])
adata4.obs['n_spliced'] = adata4.layers['spliced'].sum(1)
adata4.obs['n_unspliced'] = adata4.layers['unspliced'].sum(1)
adata4.obs['n_genes'] = (adata4.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata4.var_names.str.startswith('MT-')
adata4.obs['percent_mito'] = np.sum(
    adata4[:, mito_genes].X, axis=1) / np.sum(adata4.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata4.var_names.str.startswith('RP')
adata4.obs['percent_ribo'] = np.sum(
    adata4[:, ribo_genes].X, axis=1) / np.sum(adata4.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata4.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata4.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata4.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata4.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata4, min_counts=1600)
sc.pp.filter_cells(adata4, max_counts=27500)
adata4 = adata4[adata4.obs['n_unspliced']>900]
sc.pp.filter_cells(adata4, min_genes=500)

In [ ]:
adata4 = adata4[adata4.obs['percent_mito'] < 0.3, :]

In [ ]:
adata4 = adata4[adata4.obs['doublet'] != '1']
adata4 = adata4[adata4.obs['doublet_score'] < 0.3]

### 2.7 inflamed6 QC

In [ ]:
# Calculate QC covariates
adata6.obs['n_counts'] = adata6.X.sum(1)
adata6.obs['log_counts'] = np.log(adata6.obs['n_counts'])
adata6.obs['n_spliced'] = adata6.layers['spliced'].sum(1)
adata6.obs['n_unspliced'] = adata6.layers['unspliced'].sum(1)
adata6.obs['n_genes'] = (adata6.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata6.var_names.str.startswith('MT-')
adata6.obs['percent_mito'] = np.sum(
    adata6[:, mito_genes].X, axis=1) / np.sum(adata6.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata6.var_names.str.startswith('RP')
adata6.obs['percent_ribo'] = np.sum(
    adata6[:, ribo_genes].X, axis=1) / np.sum(adata6.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata6.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata6.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata6.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata6.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata6, min_counts=3100)
sc.pp.filter_cells(adata6, max_counts=14000)
adata6 = adata6[adata6.obs['n_unspliced']>900]
sc.pp.filter_cells(adata6, min_genes=900)

In [ ]:
adata6 = adata6[adata6.obs['percent_mito'] < 0.4, :]

In [ ]:
adata6 = adata6[adata6.obs['doublet'] != '1']
adata6 = adata6[adata6.obs['doublet_score'] < 0.3]

### 2.8 inflamed10 QC

In [ ]:
# Calculate QC covariates
adata10.obs['n_counts'] = adata10.X.sum(1)
adata10.obs['log_counts'] = np.log(adata10.obs['n_counts'])
adata10.obs['n_spliced'] = adata10.layers['spliced'].sum(1)
adata10.obs['n_unspliced'] = adata10.layers['unspliced'].sum(1)
adata10.obs['n_genes'] = (adata10.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata10.var_names.str.startswith('MT-')
adata10.obs['percent_mito'] = np.sum(
    adata10[:, mito_genes].X, axis=1) / np.sum(adata10.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata10.var_names.str.startswith('RP')
adata10.obs['percent_ribo'] = np.sum(
    adata10[:, ribo_genes].X, axis=1) / np.sum(adata10.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata10.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata10.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata10.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata10.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata10, min_counts=3500)
sc.pp.filter_cells(adata10, max_counts=17000)
adata10 = adata10[adata10.obs['n_unspliced']>500]
sc.pp.filter_cells(adata10, min_genes=800)

In [ ]:
adata10 = adata10[adata10.obs['percent_mito'] < 0.45, :]

In [ ]:
adata10 = adata10[adata10.obs['doublet'] != '1']
adata10 = adata10[adata10.obs['doublet_score'] < 0.3]

### 2.9 noninflamed2 QC 

In [ ]:
# Calculate QC covariates
adata2.obs['n_counts'] = adata2.X.sum(1)
adata2.obs['log_counts'] = np.log(adata2.obs['n_counts'])
adata2.obs['n_spliced'] = adata2.layers['spliced'].sum(1)
adata2.obs['n_unspliced'] = adata2.layers['unspliced'].sum(1)
adata2.obs['n_genes'] = (adata2.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata2.var_names.str.startswith('MT-')
adata2.obs['percent_mito'] = np.sum(
    adata2[:, mito_genes].X, axis=1) / np.sum(adata2.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata2.var_names.str.startswith('RP')
adata2.obs['percent_ribo'] = np.sum(
    adata2[:, ribo_genes].X, axis=1) / np.sum(adata2.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata2.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata2.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata2.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata2.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata2, min_counts=1800)
sc.pp.filter_cells(adata2, max_counts=25000)
adata2 = adata2[adata2.obs['n_unspliced']>1100]
sc.pp.filter_cells(adata2, min_genes=800)

In [ ]:
adata2 = adata2[adata2.obs['percent_mito'] < 0.3, :]

In [ ]:
adata2 = adata2[adata2.obs['doublet'] != '1']
adata2 = adata2[adata2.obs['doublet_score'] < 0.3]

### 2.10 noninflamed5 QC

In [ ]:
# Calculate QC covariates
adata5.obs['n_counts'] = adata5.X.sum(1)
adata5.obs['log_counts'] = np.log(adata5.obs['n_counts'])
adata5.obs['n_spliced'] = adata5.layers['spliced'].sum(1)
adata5.obs['n_unspliced'] = adata5.layers['unspliced'].sum(1)
adata5.obs['n_genes'] = (adata5.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata5.var_names.str.startswith('MT-')
adata5.obs['percent_mito'] = np.sum(
    adata5[:, mito_genes].X, axis=1) / np.sum(adata5.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata5.var_names.str.startswith('RP')
adata5.obs['percent_ribo'] = np.sum(
    adata5[:, ribo_genes].X, axis=1) / np.sum(adata5.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata5.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata5.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata5.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata5.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata5, min_counts=1500)
sc.pp.filter_cells(adata5, max_counts=20000)
adata5 = adata5[adata5.obs['n_unspliced']>700]
sc.pp.filter_cells(adata5, min_genes=400)

In [ ]:
adata5 = adata5[adata5.obs['percent_mito'] < 0.3, :]

In [ ]:
adata5 = adata5[adata5.obs['doublet'] != '1']
adata5 = adata5[adata5.obs['doublet_score'] < 0.3]

### 2.11 noninflamed9 QC

In [ ]:
# Calculate QC covariates
adata9.obs['n_counts'] = adata9.X.sum(1)
adata9.obs['log_counts'] = np.log(adata9.obs['n_counts'])
adata9.obs['n_spliced'] = adata9.layers['spliced'].sum(1)
adata9.obs['n_unspliced'] = adata9.layers['unspliced'].sum(1)
adata9.obs['n_genes'] = (adata9.X > 0).sum(1)
# Compute for each cell the fraction of counts in mito genes vs. all genes
mito_genes = adata9.var_names.str.startswith('MT-')
adata9.obs['percent_mito'] = np.sum(
    adata9[:, mito_genes].X, axis=1) / np.sum(adata9.X, axis=1)
# Compute for each cell the fraction of counts in ribosomal genes vs. all genes
ribo_genes = adata9.var_names.str.startswith('RP')
adata9.obs['percent_ribo'] = np.sum(
    adata9[:, ribo_genes].X, axis=1) / np.sum(adata9.X, axis=1)

In [ ]:
# Doublet analysis
scrub = scr.Scrublet(adata9.X, expected_doublet_rate=0.06)
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30,
                                                          mean_center=False, 
                                                          normalize_variance=False,
                                                          log_transform=True)

In [ ]:
adata9.obs['doublet_score'] = scrub.doublet_scores_obs_

In [ ]:
clf = doubletdetection.BoostClassifier(n_iters=50, use_phenograph=False, standard_scaling=True)
doublets = clf.fit(adata9.X).predict(p_thresh=1e-16, voter_thresh=0.5)

In [ ]:
adata9.obs['doublet'] = list(map(str, map(int, doublets)))

In [ ]:
sc.pp.filter_cells(adata9, min_counts=2000)
sc.pp.filter_cells(adata9, max_counts=30000)
adata9 = adata9[adata9.obs['n_unspliced']>900]
sc.pp.filter_cells(adata9, min_genes=750)

In [ ]:
adata9 = adata9[adata9.obs['percent_mito'] < 0.3, :]

In [ ]:
adata9 = adata9[adata9.obs['doublet'] != '1']
adata9 = adata9[adata9.obs['doublet_score'] < 0.3]

## 3. Preprocessing

### 3.1 Normalisation

In [ ]:
adata = adata0.concatenate(adata1, adata7, adata8, adata3, adata4, adata6, adata10, adata2, adata5, adata9,
                              batch_key='Sample', batch_categories=['healthy0', 'healthy1', 'healthy7', 'healthy8',
                                                                    'inflamed3', 'inflamed4', 'inflamed6', 'inflamed10',
                                                                    'noninflamed2', 'noninflamed5', 'noninflamed9'])

In [ ]:
# Apply workaround necessary due to bugs in code
adata.layers['spliced'] = adata.layers['spliced'].astype(float)
adata.layers['unspliced'] = adata.layers['unspliced'].astype(float)

scv.pp.filter_genes(adata, min_cells=5)
scv.pp.filter_genes(adata, min_counts=20)

scv.pp.normalize_per_cell(adata, max_proportion_per_cell=0.05)
scv.pp.log1p(adata)
adata.raw = adata

adata

### 3.2 Cell cycle

Before we can move on with regression, we need to assign a cell cycle phase to all the cells. Not only it is interesting to visulaise it later on, but we will use it to diminish differences between cycling cells. 

Specifically, we want to regress out the difference between *S phase* and *G2M phase* in order to remove cycling patterns which negatively influences the RNA velocity analysis. However, we don't want to remove all signals coming from a cell cycle phase since we are still interested in looking at progenitors of different cell lineages. 

We determine the cell cycle using a simple function checking for gene enrichment. We supply two lists of genes associated with a given phase of cell cycle (taken from the **literature**). The score is the average expression of a set of genes subtracted with the average expression of a reference set of genes. The reference set is randomly sampled from all the genes. 

`CC_difference` score is calculated as the difference between cycling and not cycling cells.  

In [ ]:
ProlifMarkers = [i for i in ProlifMarkers if i in adata.var_names]

sc.tl.score_genes(adata, ProlifMarkers, score_name='Proliferation')

In [ ]:
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)
adata.obs['CC_difference'] = adata.obs['S_score'] - adata.obs['G2M_score']

### 3.3 Regression and highly variable genes (HVGs)

As we could have seen previously there are intrinsic differences between the samples (like the sequencing depth). Since we want those samples to represent the same system in the same condition (healthy colon), we can assume that most of the variance betweeen samples is technical. We get rid of that variation by regressing **number of counts** and **CC difference**. This is done using **linear regression**.

Finally, we can select Highly Variable Genes (HVGs). The remaining analysis will be done only on them. We keep all the genes (uncorrected) in the `.raw` attribute to use them for plotting and differential expression.

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)

In [ ]:
corrected = sc.api.pp.mnn_correct(adata[adata.obs['Sample']=='healthy0'], adata[adata.obs['Sample']=='inflamed3'], 
                                  adata[adata.obs['Sample']=='noninflamed2'], adata[adata.obs['Sample']=='healthy1'],
                                  adata[adata.obs['Sample']=='inflamed4'], adata[adata.obs['Sample']=='noninflamed5'],
                                  adata[adata.obs['Sample']=='healthy7'], adata[adata.obs['Sample']=='inflamed6'],
                                  adata[adata.obs['Sample']=='noninflamed9'], adata[adata.obs['Sample']=='healthy8'],
                                  adata[adata.obs['Sample']=='inflamed10'], 
                                  batch_key='Sample', batch_categories=['healthy0', 'inflamed3', 'noninflamed2', 'healthy1',
                                                                        'inflamed4', 'noninflamed5', 'healthy7', 'inflamed6',
                                                                        'noninflamed9', 'healthy8', 'inflamed10'],
                                  var_subset=list(adata.var_names[adata.var['highly_variable']]), k=15, var_adj=True, 
                                  do_concatenate=True, save_raw=True, n_jobs=12)

In [ ]:
adata = corrected[0].copy()

In [ ]:
adata.write("all.h5ad")

In [ ]:
adata.obs['Condition'] = adata.obs['Sample']
adata.obs['Condition'] = adata.obs['Condition'].replace("healthy0", "Healthy")
adata.obs['Condition'] = adata.obs['Condition'].replace("healthy1", "Healthy")
adata.obs['Condition'] = adata.obs['Condition'].replace("healthy7", "Healthy")
adata.obs['Condition'] = adata.obs['Condition'].replace("healthy8", "Healthy")
adata.obs['Condition'] = adata.obs['Condition'].replace("inflamed3", "Ulcerated")
adata.obs['Condition'] = adata.obs['Condition'].replace("inflamed4", "Ulcerated")
adata.obs['Condition'] = adata.obs['Condition'].replace("inflamed6", "Ulcerated")
adata.obs['Condition'] = adata.obs['Condition'].replace("inflamed10", "Ulcerated")
adata.obs['Condition'] = adata.obs['Condition'].replace("noninflamed2", "Non-ulcerated")
adata.obs['Condition'] = adata.obs['Condition'].replace("noninflamed5", "Non-ulcerated")
adata.obs['Condition'] = adata.obs['Condition'].replace("noninflamed9", "Non-ulcerated")

adata.obs['Condition'] = adata.obs['Condition'].astype('category')
Condition_order = ['Healthy',
                   'Non-ulcerated',
                   'Ulcerated']
adata.obs['Condition'].cat.reorder_categories(Condition_order, inplace=True)

In [ ]:
Condition_colors = np.zeros(len(set(adata.obs['Condition'])))
Condition_colors = Condition_colors.astype('U7')

Condition_colors[[0]] = '#808080' # HEALTHY / grey
Condition_colors[[1]] = '#E5CC00'  # NONINFLAMED / yellow
Condition_colors[[2]] = '#8B0000'  # INFLAMED / red

adata.uns['Condition_colors'] = Condition_colors

We also subset our adata object to include only progenitor cells. We define those as cells being in cell cycle, so having **positive S and/or G2M scores**. We will use the progenitor dataset at a later part of the analysis.

In [ ]:
G2M_cells = [name for name in adata.obs_names if adata.obs['G2M_score'][name] > 0]
S_cells = [name for name in adata.obs_names if adata.obs['S_score'][name] > 0]
CC_cells = set(G2M_cells + S_cells)
adata_progen = adata[list(CC_cells),:].copy()

### 3.4 Dimentionality reduction and Principal Component analysis

In [ ]:
sc.pp.regress_out(adata, ['S_score', 'G2M_score'], n_jobs=12)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

...and plot it to look where is the most variance in the dataset coming from.

In [ ]:
# Change plotting settings for more visually pleasing.
scv.settings.set_figure_params('scvelo', dpi=150, vector_friendly=False)

features = ['Sample', 'Condition', 'log_counts', 'phase', 'percent_mito', 'doublet_score']
sc.pl.pca(adata, color=features, ncols=2)

In [ ]:
features = ['LGR5', 'CEACAM1', 'MUC2', 'CA1', 'BEST4', 'KRT20', 'SCGN', 'LRMP']
sc.pl.pca_overview(adata, color=features, projection='3d', ncols=2)

## 4. Graph embedding and clustering

### 4.1 Batch correction with bbknn

In [ ]:
sc.external.pp.bbknn(adata, batch_key='Sample', neighbors_within_batch=5, n_pcs=30,
                     approx=False, use_faiss=False, metric='euclidean')

### 4.2 Graph embedding

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.write("all.h5ad")

In [ ]:
sc.pl.umap(adata, color='FOS', cmap='YlOrRd')

In [ ]:
features = ['Sample', 'Condition', 'log_counts', 'phase', 'percent_mito',
            'MUC2', 'CA1', 'CA2', 'LGR5', 'KRT20', 'LRMP', 'SCGN', 
            'CEACAM1', 'TFF1' , 'PCNA', 'MKI67', 'TFF3', 'BEST4']

sc.pl.umap(adata, color=features, use_raw=True, ncols=2)

In [ ]:
sc.pl.umap(adata[adata.obs['Condition']=='HEALTHY'], color='CD74', vmax=max(adata.raw[:,'CD74'].X.todense())[0,0], cmap='YlOrRd')

In [ ]:
sc.pl.umap(adata, color=['LGR5', 'OLFM4', 'FN1'], cmap='YlOrRd')

In [ ]:
gene = 'ZFP57'
adata_qc[:,'{}'.format(gene)].X.sum()

In [ ]:
ANXA1,
CCND1, 
STMN1, RRM2,
TUBA1B, HELLS,
BASP1

In [ ]:
# Elk3, Tead4, Fos and Zfp57

sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
gene = 'ANXA1'

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10,5))

fig.tight_layout(pad=1.5)

fig.suptitle('{} expression'.format(gene), y=0.85, x=0.55)

sc.pl.umap(adata[adata.obs['Condition']=='HEALTHY'], color=['{}'.format(gene)], title='Healthy', ax=ax1, show=False, 
           size=15, cmap='YlOrRd', vmax=max(adata.raw[:,'{}'.format(gene)].X.todense())[0,0])
ax1.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='NONINFLAMED'], color=['{}'.format(gene)], title='Non-ulcerated', ax=ax2, show=False, 
           size=15, cmap='YlOrRd', vmax=max(adata.raw[:,'{}'.format(gene)].X.todense())[0,0])
ax2.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='INFLAMED'], color=['{}'.format(gene)], title='Ulcerated', ax=ax3, show=False, 
           size=15, cmap='YlOrRd', vmax=max(adata.raw[:,'{}'.format(gene)].X.todense())[0,0])
ax3.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

fig.savefig('figures/all_compare_{}.pdf'.format(gene), bbox_inches='tight')

#ax=sc.pl.violin(adata, '{}'.format(gene), groupby='Condition', scale='count', palette=['#808080', '#E5CC00', '#8B0000'], 
#             rotation=45, size=0.1, show=False, alpha=0.1, inner='box', stripplot=False, cut=0)
#ax.grid(False)
#ax.set_xticklabels(['HEALTHY', 'NON-ULCERATED', 'ULCERATED'])
#plt.savefig('figures/{}_violin.pdf'.format(gene), bbox_inches = "tight")
#

# Expression summary 

In [ ]:
t = pd.DataFrame(data=adata.raw.X.toarray(), index=adata.obs_names, columns=adata.raw.var_names)
Expression_OLFM4 = {'Condition':adata.obs['Condition'], 'Celltype':adata.obs['Celltype'], 'Expression':t['OLFM4']}


Expression_OLFM4 = pd.DataFrame(data=Expression_OLFM4)

In [ ]:
Expression_OLFM4.to_csv('Expression_OLFM4.csv')

In [ ]:
t = pd.DataFrame(data=adata.raw.X.toarray(), index=adata.obs_names, columns=adata.raw.var_names)
Expression_CD74 = {'Condition':adata.obs['Condition'], 'Celltype':adata.obs['Celltype'], 'Expression':t['CD74']}


Expression_CD74 = pd.DataFrame(data=Expression_CD74)

In [ ]:
Expression_CD74.to_csv('Expression_CD74.csv')

Using a list of genes taken from the literature we can also determine which cells are proliferating. 

In [ ]:
sc.pl.umap(adata, color='Proliferation', title="Proliferation score")

In [ ]:
sc.pl.umap(adata, color=['ATOH1', 'SOX4'], size=30)

In [ ]:
sc.pl.umap(adata, color='ADH1C')

### 4.3 Finding clusters and celltypes 

We look for clusters in the cell graph using unsupervised clustering method `leiden` (better than louvain).

In [ ]:
sc.tl.leiden(adata, resolution=1.7)

In [ ]:
sc.tl.leiden(adata, resolution=0.4, restrict_to=('leiden', ['0']))

In [ ]:
sc.tl.leiden(adata, resolution=0.15, restrict_to=('leiden_R', ['16']))

In [ ]:
sc.pl.umap(adata, color='leiden_R', legend_loc='on data')

In [ ]:
sc.pl.umap(adata, color='SOX4', legend_loc='on data', cmap='YlOrRd')

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.25, 
                               max_out_group_fraction=0.75, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

Following expression of the marker genes we can reassign and rename clusters to more meaningful structures that should correspond to celltypes. 

In [ ]:
adata.obs['Celltype'] = adata.obs['leiden_R']
adata.obs['Celltype'] = adata.obs['Celltype'].replace("0,0", "Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("0,1", "Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("0,2", "TA Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("1", "TA Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("2", "Stem")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("3", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("4", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("5", "TA BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("6", "Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("7", "Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("8", "Stem")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("9", "BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("10", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("11", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("12", "CT BEST4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("13", "Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("14", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("15", "TA SOX4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("16,0", "Tuft")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("16,1", "TA SOX4+")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("16,2", "Tuft")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("17", "TA Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("18", "TA Goblet")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("19", "CT Colonocytes")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("20", "EECs")
adata.obs['Celltype'] = adata.obs['Celltype'].replace("21", "CT Goblet")

adata.obs['Celltype'] = adata.obs['Celltype'].astype('category')
Celltype_order = ['Stem',
                  'TA SOX4+',
                  'TA Colonocytes', 'Colonocytes', 'CT Colonocytes',
                  'TA BEST4+', 'BEST4+', 'CT BEST4+', 
                  'TA Goblet', 'Goblet', 'CT Goblet',                  
                  'EECs' ,'Tuft']
adata.obs['Celltype'].cat.reorder_categories(Celltype_order, inplace=True)

Assign colors to celltypes

In [ ]:
vega_colors = np.array(sc.pl.palettes.vega_20_scanpy)

celltype_colors = np.zeros(len(set(adata.obs['Celltype'])))
celltype_colors = celltype_colors.astype('U7')

celltype_colors[[0]] =  vega_colors[[1]] # Stem color / orange
celltype_colors[[1]] = '#FFCC00' # TA Goblet SOX4+ colors / yellow
celltype_colors[[2, 3, 4]] = vega_colors[[12, 10, 15]]  # Colono colors / reds
celltype_colors[[5, 6, 7]] = vega_colors[[0, 8, 17]]  # BEST4 colors / blues
celltype_colors[[8, 9, 10]] = vega_colors[[2, 7, 11]]  # Goblet colors / greens
celltype_colors[[11]] = '#E53333'  # EECs / red
celltype_colors[[12]] = '#A9A9A9'  # Tuft / grey

adata.uns['Celltype_colors'] = celltype_colors

In [ ]:
adata.write("all.h5ad")

Plot celltypes with new coloring

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)


In [ ]:
sc.pl.umap(adata, color='Celltype', legend_loc='right margin', alpha=0.7, size=10, title='', save='allCelltype')

In [ ]:
adata.obs['Condition'] = adata.obs['Condition'].replace("NONINFLAMED", "NON-ULCERATED")
adata.obs['Condition'] = adata.obs['Condition'].replace("INFLAMED", "ULCERATED")

In [ ]:
sc.pl.umap(adata, color='Condition', legend_loc='right margin', save='allCondition', alpha=0.5, size=10, title='')

Differential expression analysis between celltypes

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon')
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.5, 
                               max_out_group_fraction=0.75, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

In [ ]:
dataset_sizes = adata.obs['Sample'].value_counts()

celltype_summary = pd.DataFrame(adata.obs.groupby(['Condition', 'Sample'])['Celltype'].value_counts()).rename(columns={'Celltype': "Counts"})

In [ ]:
celltype_summary = celltype_summary.reset_index()

celltype_summary['Percentage'] = [celltype_summary.loc[index, 'Counts']*100/dataset_sizes[celltype_summary.loc[index, 'Sample']] 
                                  for index in celltype_summary.index]

celltype_summary.to_csv('all_celltype_summary.csv')

In [ ]:
celltype_summary

## Cell cycle comparison

In [ ]:
sample_celltype_sizes = adata.obs.groupby(['Sample'])['Celltype'].value_counts()
phase_summary = pd.DataFrame(adata.obs.groupby(['Condition', 'Sample', 'Celltype'])['phase'].value_counts()).rename(columns={'phase': "Counts"})

In [ ]:
phase_summary = phase_summary.reset_index()

phase_summary['Percentage'] = [phase_summary.loc[index, 'Counts']*100/sample_celltype_sizes[phase_summary.loc[index, 'Sample'], phase_summary.loc[index, 'Celltype']] 
                                  for index in phase_summary.index]

phase_summary.to_csv('all_phase_summary.csv')

## 4.4 Correlation matrix plot

In [ ]:
adata.obs['CD74_expression'] = adata.raw[:,'CD74'].X.todense()

In [ ]:
cats = []
types = list(set(adata.obs['Celltype']))
conds = list(set(adata.obs['Condition']))

for i in range(len(conds)):
    cond = conds[i]
    if cond == 'Non-ulcerated':
        cat = (typ + ":" + cond + ' CD74+' for typ in types)
        cats.extend(cat)
        cat = (typ + ":" + cond + ' CD74-' for typ in types)
        cats.extend(cat)
    else:
        cat = (typ + ":" + cond for typ in types)
        cats.extend(cat)

In [ ]:
adata.obs['CelltypeCondition'] = adata.obs['Celltype']
adata.obs['CelltypeCondition'] = adata.obs['CelltypeCondition'].cat.add_categories(cats)
for i in range(len(adata.obs)):
    if adata.obs['Condition'][i] == 'Non-ulcerated':
        if adata.obs['CD74_expression'][i] > 0.3970:
            adata.obs['CelltypeCondition'][i] = adata.obs['Celltype'][i] + ":" + adata.obs['Condition'][i] + ' CD74+'
        else:
            adata.obs['CelltypeCondition'][i] = adata.obs['Celltype'][i] + ":" + adata.obs['Condition'][i] + ' CD74-'
    else:
        adata.obs['CelltypeCondition'][i] = adata.obs['Celltype'][i] + ":" + adata.obs['Condition'][i]

adata.obs['CelltypeCondition'] = adata.obs['CelltypeCondition'].values.remove_unused_categories()

In [ ]:
adata.obs['ConditionSplit'] = adata.obs['Condition']
adata.obs['ConditionSplit'] = adata.obs['ConditionSplit'].cat.add_categories(['Non-ulcerated CD74+', 'Non-ulcerated CD74-'])
for i in range(len(adata.obs)):
    if adata.obs['ConditionSplit'][i] == 'Non-ulcerated':
        if adata.obs['CD74_expression'][i] > 0.3970:
            adata.obs['ConditionSplit'][i] = adata.obs['ConditionSplit'][i] + ' CD74+'
        else:
            adata.obs['ConditionSplit'][i] = adata.obs['ConditionSplit'][i] + ' CD74-'

adata.obs['ConditionSplit'] = adata.obs['ConditionSplit'].values.remove_unused_categories()

In [ ]:
adata.obs['ConditionSplit']

In [ ]:
sc.tl.dendrogram(adata, 'ConditionSplit', n_pcs=50, cor_method = 'pearson', optimal_ordering = True)

In [ ]:
sc.pl.correlation_matrix(adata, 'ConditionSplit', show_correlation_numbers = True)

In [ ]:
adata_regress = adata.copy()

In [ ]:
adata_regress.obsm['X_pca'].shape

In [ ]:
adata_regress.obsm['X_pca'] = np.delete(adata_regress.obsm['X_pca'], 0, 1)
adata_regress.obsm['X_pca'] = np.delete(adata_regress.obsm['X_pca'], 1, 1)
adata_regress.obsm['X_pca'].shape

In [ ]:
sc.tl.dendrogram(adata_regress, 'ConditionSplit', n_pcs=48, cor_method = 'pearson', optimal_ordering = True)

In [ ]:
sc.pl.correlation_matrix(adata_regress, 'ConditionSplit', show_correlation_numbers = True)

In [ ]:
sc.pl.pca_variance_ratio(adata, log=False)

In [ ]:
sc.pl.pca_loadings(adata, components = '1,2,3,4,5,6, 7, 8, 9, 10, 11, 12, 13, 14, 15', include_lowest=False)

In [ ]:
sc.pl.umap(adata, color='Condition', cmap='YlOrRd')

In [ ]:
adata.obs['PC1'] = adata.obsm['X_pca'][:, 0]
adata.obs['PC2'] = adata.obsm['X_pca'][:, 1]
adata.obs['PC3'] = adata.obsm['X_pca'][:, 2]
adata.obs['PC4'] = adata.obsm['X_pca'][:, 3]
adata.obs['PC5'] = adata.obsm['X_pca'][:, 4]
adata.obs['PC6'] = adata.obsm['X_pca'][:, 5]
adata.obs['PC7'] = adata.obsm['X_pca'][:, 6]
adata.obs['PC8'] = adata.obsm['X_pca'][:, 7]
adata.obs['PC9'] = adata.obsm['X_pca'][:, 8]
adata.obs['PC10'] = adata.obsm['X_pca'][:, 9]
adata.obs['PC11'] = adata.obsm['X_pca'][:, 10]
adata.obs['PC12'] = adata.obsm['X_pca'][:, 11]
adata.obs['PC13'] = adata.obsm['X_pca'][:, 12]
adata.obs['PC14'] = adata.obsm['X_pca'][:, 13]
adata.obs['PC15'] = adata.obsm['X_pca'][:, 14]

In [ ]:
pcs = ['PC' + str(i) for i in range(1,16)]

sc.pl.umap(adata, color=pcs, cmap='bwr')

### 5.1 PAGA

Redo clustering with higher resolution.

In [ ]:
adata.obs['SubCelltype'] = adata.obs['Celltype']
for celltype in adata.obs['Celltype'].cat.categories.tolist():
    sc.tl.leiden(adata, resolution=1, restrict_to=('SubCelltype', [celltype]), key_added='SubCelltype')

In [ ]:
sc.pl.umap(adata, color='SubCelltype', legend_loc='right margin', legend_fontweight='light')#, save='test1')

Run PAGA on smaller (higher resolution) clusters, while keeping the information about the celltypes (coloring).

In [ ]:
sc.tl.paga(adata, groups='SubCelltype')

In [ ]:
sc.pl.paga(adata, color='Celltype', threshold=1, 
           node_size_scale=0.6, edge_width_scale=0.1, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0}, frameon=False)#, save='all')

In [ ]:
sc.pl.paga(adata, color='Celltype', threshold=0.9, 
           node_size_scale=0.6, edge_width_scale=0.1, node_size_power=0.2, 
           layout='fr', text_kwds={'alpha':0})

### 5.2 Graph embedding based on PAGA

In [ ]:
#sc.set_figure_params(fontsize=15, vector_friendly=False, dpi=150)
scv.settings.set_figure_params(dpi=150, vector_friendly=False)

In [ ]:
sc.tl.draw_graph(adata, init_pos='paga')

In [ ]:
sc.pl.draw_graph(adata, color='Celltype', 
                 size=20, legend_fontsize=6, frameon=False, edges=False, save='all')

In [ ]:
sc.pl.draw_graph(adata, color='Celltype', 
                 size=20, legend_fontsize=6, frameon=False, edges=True, save='alledges')

In [ ]:
sc.pl.draw_graph(adata, color='SOX4',
                 size=30, legend_fontsize=6, frameon=False, edges=False)

In [ ]:
sc.pl.draw_graph(adata, color='ATOH1',
                 size=30, legend_fontsize=6, frameon=False, edges=False)

### 5.3 Diffusion-based pseudotime

In [ ]:
# Set one of the stem cells as the starting point for the pseudotime.
# Biased - compare with latent time. 
adata.uns['iroot'] = np.flatnonzero(adata.obs['Celltype']  == 'Stem')[0]

sc.tl.diffmap(adata)
sc.tl.dpt(adata)

In [ ]:
scv.pl.scatter(adata, color='dpt_pseudotime', legend_loc='on data', 
                 size=20, fontsize=24, frameon=False, 
                 cmap='viridis', title='Pseudotime')

## Load the data

In [ ]:
adata = sc.read('all.h5ad')

In [ ]:
adata

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)

# 6,5. Embedding density

In [ ]:
sc.tl.embedding_density(adata, basis='umap', groupby='Condition')

In [ ]:
sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
sc.pl.embedding_density(adata, basis='umap', key='umap_density_Condition', group=['HEALTHY', 'NONINFLAMED', 'INFLAMED'], 
                        save='all')

## 8. GO analysis

In [ ]:
from goatools.base import download_go_basic_obo
obo_fname = download_go_basic_obo()

In [ ]:
from goatools.base import download_ncbi_associations
fin_gene2go = download_ncbi_associations()

In [ ]:
from goatools.obo_parser import GODag

obodag = GODag("go-basic.obo")

In [ ]:
from goatools.anno.genetogo_reader import Gene2GoReader

# Read NCBI's gene2go. Store annotations in a list of namedtuples
objanno = Gene2GoReader(fin_gene2go, taxids=[9606])

# Get namespace2association where:
#    namespace is:
#        BP: biological_process               
#        MF: molecular_function
#        CC: cellular_component
#    assocation is a dict:
#        key: NCBI GeneID
#        value: A set of GO IDs associated with that gene
ns2assoc = objanno.get_ns2assc()

for nspc, id2gos in ns2assoc.items():
    print("{NS} {N:,} annotated human genes".format(NS=nspc, N=len(id2gos)))

In [ ]:
# Use all genes as background genes
adata_qc
BackGroundGenes = adata_qc.var_names

In [ ]:
import mygene
mg = mygene.MyGeneInfo()

In [ ]:
BackGroundGenesEntrez = mg.querymany(BackGroundGenes, scopes='symbol', fields='entrezgene', species='human')

In [ ]:
# Extract just the entrez names that were found in the database (not NA)
BackGroundGenesEntrezValues = list(pd.DataFrame(BackGroundGenesEntrez)['entrezgene'].dropna().astype(int))

In [ ]:
print(len(BackGroundGenes))
print(len(BackGroundGenesEntrezValues))

In [ ]:
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

goeaobj = GOEnrichmentStudyNS(
        BackGroundGenesEntrezValues, # List of background human genes
        ns2assoc, # geneid/GO associations
        obodag, # Ontologies
        propagate_counts = False,
        alpha = 0.05, # default significance cut-off
        methods = ['fdr_bh']) # defult multipletest correction method

### 8.1 GO terms for crypt top populations 

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT Goblet'], reference='Goblet', n_genes=10000)
deGobletCT = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT Colonocytes'], reference='Colonocytes', n_genes=10000)
deColonocytesCT = adata.uns['rank_genes_groups']

sc.tl.rank_genes_groups(adata, groupby='Celltype', method='wilcoxon', groups=['CT BEST4+'], reference='BEST4+', n_genes=10000)
deBEST4CT = adata.uns['rank_genes_groups']

In [ ]:
deGobletCTdf = pd.DataFrame(data={'Name':deGobletCT['names'].astype(str),
                                  'Score':deGobletCT['scores'].astype(float),
                                  'log2FC':deGobletCT['logfoldchanges'].astype(float),
                                  'p-value':deGobletCT['pvals_adj'].astype(float)},
                            index=list(range(len(deGobletCT['names']))))
deGobletCTdf = deGobletCTdf[deGobletCTdf['log2FC']>0]
deGobletCTdf = deGobletCTdf[deGobletCTdf['p-value']<0.05]

In [ ]:
deColonocytesCTdf = pd.DataFrame(data={'Name':deColonocytesCT['names'].astype(str),
                                  'Score':deColonocytesCT['scores'].astype(float),
                                  'log2FC':deColonocytesCT['logfoldchanges'].astype(float),
                                  'p-value':deColonocytesCT['pvals_adj'].astype(float)},
                            index=list(range(len(deColonocytesCT['names']))))
deColonocytesCTdf = deColonocytesCTdf[deColonocytesCTdf['log2FC']>0]
deColonocytesCTdf = deColonocytesCTdf[deColonocytesCTdf['p-value']<0.05]

In [ ]:
deBEST4CTdf = pd.DataFrame(data={'Name':deBEST4CT['names'].astype(str),
                                  'Score':deBEST4CT['scores'].astype(float),
                                  'log2FC':deBEST4CT['logfoldchanges'].astype(float),
                                  'p-value':deBEST4CT['pvals_adj'].astype(float)},
                            index=list(range(len(deBEST4CT['names']))))
deBEST4CTdf = deBEST4CTdf[deBEST4CTdf['log2FC']>0]
deBEST4CTdf = deBEST4CTdf[deBEST4CTdf['p-value']<0.05]

In [ ]:
print(len(deGobletCTdf['Name']))
print(len(deColonocytesCTdf['Name']))
print(len(deBEST4CTdf['Name']))

In [ ]:
CTgenes = set(deGobletCTdf['Name']).intersection(deColonocytesCTdf['Name'], deBEST4CTdf['Name'])

In [ ]:
CTgenes = set(deGobletCTdf['Name'])

In [ ]:
CTgenes = set(deColonocytesCTdf['Name'])

In [ ]:
CTgenes = set(deBEST4CTdf['Name'])

In [ ]:
data = "BEST4"

In [ ]:
# Turn gene symbols into Entrez IDs
CTgenesEntrez = mg.querymany(CTgenes, scopes='symbol', fields='entrezgene', species='human')
CTgenesEntrezValues = list(pd.DataFrame(CTgenesEntrez)['entrezgene'].dropna().astype(int))
CTgenesEntrezDict = dict(zip(list(pd.DataFrame(CTgenesEntrez)['entrezgene']), list(pd.DataFrame(CTgenesEntrez)['query'])))

In [ ]:
print(len(CTgenes))
print(len(CTgenesEntrezValues))

In [ ]:
# Run the GOEA
CT_goea_results_all = goeaobj.run_study(CTgenesEntrezValues)
CT_goea_results_sig = [r for r in CT_goea_results_all if r.p_fdr_bh < 0.05]

In [ ]:
goeaobj.wr_txt("{}goterms.txt".format(data), CT_goea_results_sig)

In [ ]:
from goatools.godag_plot import plot_gos, plot_results, plot_goid2goobj

plot_results("BEST4CTgoterms{NS}.pdf", CT_goea_results_sig)

In [ ]:
goid_subset = [
    'GO:0043312', # neutrophil degranulation
    'GO:0038096', # Fc-gamma receptor signaling pathway involved in phagocytosis
    'GO:0043123' # NF-kappaB
]
plot_gos("{}CTgoterms_immune.pdf".format(data),
    goid_subset, # Source GO ids
    obodag, 
    goea_results=CT_goea_results_sig) # Use pvals for coloring

In [ ]:
# change entrez into symbol
entrez = [387, 567, 634, 967, 978, 1476, 2495, 2517, 2934, 3728, 3958, 4680, 5646, 5660, 5879, 5906, 7879, 8649, 8655, 8673, 9218, 9798, 10487, 11240, 23593, 26578, 51316, 51382, 51646, 51719, 54509, 64114]
symbolsCT = mg.querymany(entrez, scopes='entrezgene', fields='symbol', species='human')
symbolsCT = list(pd.DataFrame(symbolsCT)['symbol'])
print(symbolsCT)

In [ ]:
sc.pl.umap(adata, color=symbolsCT, legend_loc='right margin', alpha=0.7, size=10, save='ColonoDegranulo')

### 8.2 DE and GO terms for stem cells

In [ ]:
adata.obs['ConditionCelltype'] = adata.obs['Celltype']
adata.obs['ConditionCelltype'] = adata.obs['ConditionCelltype'].cat.add_categories(['HealthyStem', 'Non-ulceratedStem', 'UlceratedStem'])
for i in range(len(adata.obs)):
    if adata.obs['Celltype'][i] == 'Stem':
        if adata.obs['Condition'][i] == 'Healthy':
            adata.obs['ConditionCelltype'][i] = "HealthyStem"
        elif adata.obs['Condition'][i] == 'Non-ulcerated':
            adata.obs['ConditionCelltype'][i] = "Non-ulceratedStem"
        else: 
            adata.obs['ConditionCelltype'][i] = "UlceratedStem"    
adata.obs['ConditionCelltype'] = adata.obs['ConditionCelltype'].values.remove_unused_categories()

In [ ]:
sc.pl.umap(adata, color='ConditionCelltype')

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='ConditionCelltype', method='wilcoxon', groups=['Non-ulceratedStem'], reference='HealthyStem', n_genes=10000)
deNoninflamed = adata.uns['rank_genes_groups']
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.5, 
                               max_out_group_fraction=0.5, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

sc.tl.rank_genes_groups(adata, groupby='ConditionCelltype', method='wilcoxon', groups=['UlceratedStem'], reference='HealthyStem', n_genes=10000)
deInflamed = adata.uns['rank_genes_groups']
sc.tl.filter_rank_genes_groups(adata, min_in_group_fraction=0.5, 
                               max_out_group_fraction=0.5, min_fold_change=1.5)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, key='rank_genes_groups_filtered', 
                                standard_scale='var', color_map='bwr', dendrogram=False)

In [ ]:
lst = ['HLA-DRA', 'CD74', 'OLFM4', 'FN1', 'LCN2', 'HLA-DMA', 'HLA-DRB1', 'MT-ATP8', 'CCL20', 'PTPRO']
sc.pl.umap(adata, color=lst, cmap='YlOrRd')

In [ ]:
# Elk3, Tead4, Fos and Zfp57

sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
gene = lst[9]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10,5))

fig.tight_layout(pad=1.5)

fig.suptitle('{} expression'.format(gene), y=0.85, x=0.55)

sc.pl.umap(adata[adata.obs['Condition']=='Healthy'], color=['{}'.format(gene)], title='Healthy', ax=ax1, show=False, 
           size=15, cmap='YlOrRd', vmax=max(adata.raw[:,'{}'.format(gene)].X.todense())[0,0])
ax1.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='Non-ulcerated'], color=['{}'.format(gene)], title='Non-ulcerated', ax=ax2, show=False, 
           size=15, cmap='YlOrRd', vmax=max(adata.raw[:,'{}'.format(gene)].X.todense())[0,0])
ax2.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='Ulcerated'], color=['{}'.format(gene)], title='Ulcerated', ax=ax3, show=False, 
           size=15, cmap='YlOrRd', vmax=max(adata.raw[:,'{}'.format(gene)].X.todense())[0,0])
ax3.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

In [ ]:
deNoninflameddf = pd.DataFrame(data={'Name':deNoninflamed['names'].astype(str),
                                  'Score':deNoninflamed['scores'].astype(float),
                                  'log2FC':deNoninflamed['logfoldchanges'].astype(float),
                                  'p-value':deNoninflamed['pvals_adj'].astype(float)},
                            index=list(range(len(deNoninflamed['names']))))
deNoninflameddf = deNoninflameddf[deNoninflameddf['log2FC']>0.5]
deNoninflameddf = deNoninflameddf[deNoninflameddf['p-value']<0.05]

deInflameddf = pd.DataFrame(data={'Name':deInflamed['names'].astype(str),
                                  'Score':deInflamed['scores'].astype(float),
                                  'log2FC':deInflamed['logfoldchanges'].astype(float),
                                  'p-value':deInflamed['pvals_adj'].astype(float)},
                            index=list(range(len(deInflamed['names']))))
deInflameddf = deInflameddf[deInflameddf['log2FC']>0.5]
deInflameddf = deInflameddf[deInflameddf['p-value']<0.05]

In [ ]:
print(len(deNoninflameddf['Name']))
print(len(deInflameddf['Name']))

In [ ]:
deNoninflameddf

In [ ]:
DEgenes = set(deNoninflameddf['Name'])

In [ ]:
DEgenes = set(deInflameddf['Name'])

In [ ]:
DEgenes = list(set(deInflameddf['Name']) & set(deNoninflameddf['Name']))

In [ ]:
len(DEgenes)

In [ ]:
data = "StemCommonUP"

In [ ]:
# Turn gene symbols into Entrez IDs
DEgenesEntrez = mg.querymany(DEgenes, scopes='symbol', fields='entrezgene', species='human')
DEgenesEntrezValues = list(pd.DataFrame(DEgenesEntrez)['entrezgene'].dropna().astype(int))
DEgenesEntrezDict = dict(zip(list(pd.DataFrame(DEgenesEntrez)['entrezgene']), list(pd.DataFrame(DEgenesEntrez)['query'])))

In [ ]:
print(len(DEgenes))
print(len(DEgenesEntrezValues))

In [ ]:
pd.DataFrame(DEgenesEntrezValues).to_csv('DEGsStemCommonUP.csv')

In [ ]:
pd.DataFrame(BackGroundGenesEntrezValues).to_csv('DEGsAllAll.csv')

In [ ]:
# Run the GOEA
DE_goea_results_all = goeaobj.run_study(DEgenesEntrezValues)
DE_goea_results_sig = [r for r in DE_goea_results_all if r.p_fdr_bh < 0.05]

In [ ]:
deInflameddf.to_csv('StemUlceratedDEgenesUP.csv')

In [ ]:
deNoninflameddf.to_csv('StemNon-ulceratedDEgenesUP.csv')

In [ ]:
goeaobj.wr_txt("{}goterms.txt".format(data), DE_goea_results_sig)
goeaobj.wr_xlsx("{}goterms.xlsx".format(data), DE_goea_results_sig)

In [ ]:
from goatools.godag_plot import plot_gos, plot_results, plot_goid2goobj

plot_results("StemInflamedgoterms{NS}.pdf", DE_goea_results_sig)

In [ ]:
# change entrez into symbol
entrez = [267, 1500, 1649, 1958, 3065, 4192, 5204, 5684, 5685, 5695, 5696, 5698, 5700, 5701, 5706, 5716, 5720, 5721, 5800, 6662, 6934, 9978, 55681, 57680]
symbolsCT = mg.querymany(entrez, scopes='entrezgene', fields='symbol', species='human')
symbolsCT = list(pd.DataFrame(symbolsCT)['symbol'])
print(symbolsCT)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['Stem:Ulcerated'], reference='Stem:Healthy', n_genes=10000)
deStem = adata.uns['rank_genes_groups']
deStem = pd.DataFrame(data={'Name':deNoninflamed['names'].astype(str),
                                  'Score':deNoninflamed['scores'].astype(float),
                                  'log2FC':deNoninflamed['logfoldchanges'].astype(float),
                                  'p-value':deNoninflamed['pvals_adj'].astype(float)},
                            index=list(range(len(deNoninflamed['names']))))
deStem = deStem[deStem['log2FC']>0.5]
deStem = deStem[deStem['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['TA SOX4+:Ulcerated'], reference='TA SOX4+:Healthy', n_genes=10000)
deTASOX4 = adata.uns['rank_genes_groups']
deTASOX4 = pd.DataFrame(data={'Name':deTASOX4['names'].astype(str),
                                  'Score':deTASOX4['scores'].astype(float),
                                  'log2FC':deTASOX4['logfoldchanges'].astype(float),
                                  'p-value':deTASOX4['pvals_adj'].astype(float)},
                            index=list(range(len(deTASOX4['names']))))
deTASOX4 = deTASOX4[deTASOX4['log2FC']>0.5]
deTASOX4 = deTASOX4[deTASOX4['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['TA Colonocytes:Ulcerated'], reference='TA Colonocytes:Healthy', n_genes=10000)
deTAColonocytes = adata.uns['rank_genes_groups']
deTAColonocytes = pd.DataFrame(data={'Name':deTAColonocytes['names'].astype(str),
                                  'Score':deTAColonocytes['scores'].astype(float),
                                  'log2FC':deTAColonocytes['logfoldchanges'].astype(float),
                                  'p-value':deTAColonocytes['pvals_adj'].astype(float)},
                            index=list(range(len(deTAColonocytes['names']))))
deTAColonocytes = deTAColonocytes[deTAColonocytes['log2FC']>0.5]
deTAColonocytes = deTAColonocytes[deTAColonocytes['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['Colonocytes:Ulcerated'], reference='Colonocytes:Healthy', n_genes=10000)
deColonocytes = adata.uns['rank_genes_groups']
deColonocytes = pd.DataFrame(data={'Name':deColonocytes['names'].astype(str),
                                  'Score':deColonocytes['scores'].astype(float),
                                  'log2FC':deColonocytes['logfoldchanges'].astype(float),
                                  'p-value':deColonocytes['pvals_adj'].astype(float)},
                            index=list(range(len(deColonocytes['names']))))
deColonocytes = deColonocytes[deColonocytes['log2FC']>0.5]
deColonocytes = deColonocytes[deColonocytes['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['CT Colonocytes:Ulcerated'], reference='CT Colonocytes:Healthy', n_genes=10000)
deCTColonocytes = adata.uns['rank_genes_groups']
deCTColonocytes = pd.DataFrame(data={'Name':deCTColonocytes['names'].astype(str),
                                  'Score':deCTColonocytes['scores'].astype(float),
                                  'log2FC':deCTColonocytes['logfoldchanges'].astype(float),
                                  'p-value':deCTColonocytes['pvals_adj'].astype(float)},
                            index=list(range(len(deCTColonocytes['names']))))
deCTColonocytes = deCTColonocytes[deCTColonocytes['log2FC']>0.5]
deCTColonocytes = deCTColonocytes[deCTColonocytes['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['TA BEST4+:Ulcerated'], reference='TA BEST4+:Healthy', n_genes=10000)
deTABEST4 = adata.uns['rank_genes_groups']
deTABEST4 = pd.DataFrame(data={'Name':deTABEST4['names'].astype(str),
                                  'Score':deTABEST4['scores'].astype(float),
                                  'log2FC':deTABEST4['logfoldchanges'].astype(float),
                                  'p-value':deTABEST4['pvals_adj'].astype(float)},
                            index=list(range(len(deTABEST4['names']))))
deTABEST4 = deTABEST4[deTABEST4['log2FC']>0.5]
deTABEST4 = deTABEST4[deTABEST4['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['BEST4+:Ulcerated'], reference='BEST4+:Healthy', n_genes=10000)
deBEST4 = adata.uns['rank_genes_groups']
deBEST4 = pd.DataFrame(data={'Name':deBEST4['names'].astype(str),
                                  'Score':deBEST4['scores'].astype(float),
                                  'log2FC':deBEST4['logfoldchanges'].astype(float),
                                  'p-value':deBEST4['pvals_adj'].astype(float)},
                            index=list(range(len(deBEST4['names']))))
deBEST4 = deBEST4[deBEST4['log2FC']>0.5]
deBEST4 = deBEST4[deBEST4['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['CT BEST4+:Ulcerated'], reference='CT BEST4+:Healthy', n_genes=10000)
deCTBEST4 = adata.uns['rank_genes_groups']
deCTBEST4 = pd.DataFrame(data={'Name':deCTBEST4['names'].astype(str),
                                  'Score':deCTBEST4['scores'].astype(float),
                                  'log2FC':deCTBEST4['logfoldchanges'].astype(float),
                                  'p-value':deCTBEST4['pvals_adj'].astype(float)},
                            index=list(range(len(deCTBEST4['names']))))
deCTBEST4 = deCTBEST4[deCTBEST4['log2FC']>0.5]
deCTBEST4 = deCTBEST4[deCTBEST4['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['TA Goblet:Ulcerated'], reference='TA Goblet:Healthy', n_genes=10000)
deTAGoblet = adata.uns['rank_genes_groups']
deTAGoblet = pd.DataFrame(data={'Name':deTAGoblet['names'].astype(str),
                                  'Score':deTAGoblet['scores'].astype(float),
                                  'log2FC':deTAGoblet['logfoldchanges'].astype(float),
                                  'p-value':deTAGoblet['pvals_adj'].astype(float)},
                            index=list(range(len(deTAGoblet['names']))))
deTAGoblet = deTAGoblet[deTAGoblet['log2FC']>0.5]
deTAGoblet = deTAGoblet[deTAGoblet['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['Goblet:Ulcerated'], reference='Goblet:Healthy', n_genes=10000)
deGoblet = adata.uns['rank_genes_groups']
deGoblet = pd.DataFrame(data={'Name':deGoblet['names'].astype(str),
                                  'Score':deGoblet['scores'].astype(float),
                                  'log2FC':deGoblet['logfoldchanges'].astype(float),
                                  'p-value':deGoblet['pvals_adj'].astype(float)},
                            index=list(range(len(deGoblet['names']))))
deGoblet = deGoblet[deGoblet['log2FC']>0.5]
deGoblet = deGoblet[deGoblet['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['CT Goblet:Ulcerated'], reference='CT Goblet:Healthy', n_genes=10000)
deCTGoblet = adata.uns['rank_genes_groups']
deCTGoblet = pd.DataFrame(data={'Name':deCTGoblet['names'].astype(str),
                                  'Score':deCTGoblet['scores'].astype(float),
                                  'log2FC':deCTGoblet['logfoldchanges'].astype(float),
                                  'p-value':deCTGoblet['pvals_adj'].astype(float)},
                            index=list(range(len(deCTGoblet['names']))))
deCTGoblet = deCTGoblet[deCTGoblet['log2FC']>0.5]
deCTGoblet = deCTGoblet[deCTGoblet['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['EECs:Ulcerated'], reference='EECs:Healthy', n_genes=10000)
deEECs = adata.uns['rank_genes_groups']
deEECs = pd.DataFrame(data={'Name':deEECs['names'].astype(str),
                                  'Score':deEECs['scores'].astype(float),
                                  'log2FC':deEECs['logfoldchanges'].astype(float),
                                  'p-value':deEECs['pvals_adj'].astype(float)},
                            index=list(range(len(deEECs['names']))))
deEECs = deEECs[deEECs['log2FC']>0.5]
deEECs = deEECs[deEECs['p-value']<0.05]

sc.tl.rank_genes_groups(adata, groupby='CelltypeCondition', method='wilcoxon', groups=['Tuft:Ulcerated'], reference='Tuft:Healthy', n_genes=10000)
deTuft = adata.uns['rank_genes_groups']
deTuft = pd.DataFrame(data={'Name':deTuft['names'].astype(str),
                                  'Score':deTuft['scores'].astype(float),
                                  'log2FC':deTuft['logfoldchanges'].astype(float),
                                  'p-value':deTuft['pvals_adj'].astype(float)},
                            index=list(range(len(deTuft['names']))))
deTuft = deTuft[deTuft['log2FC']>0.5]
deTuft = deTuft[deTuft['p-value']<0.05]

In [ ]:
print(len(deStem))
print(len(deTASOX4))
print(len(deTAColonocytes))
print(len(deColonocytes))
print(len(deCTColonocytes))
print(len(deTABEST4))
print(len(deBEST4))
print(len(deCTBEST4))
print(len(deEECs))
print(len(deTuft))

In [ ]:
deCTColonocytes

## Raul

In [ ]:
Rgenes = ['Celltype','ATF3', 'ATF4', 'HOXB13']

sc.pl.umap(adata, color=Rgenes, use_raw=True, ncols=2, cmap='RdBu_r', save='RaulColonHealthy', legend_loc='on data', size=30, alpha=0.75)

In [ ]:
sc.pl.dotplot(adata, Rgenes[1:], groupby='Celltype', var_group_rotation=0, dendrogram=False, color_map='bwr', standard_scale='var', save='RaulColonHealthyDotplot', use_raw=True)

In [ ]:
sc.tl.dendrogram(adata, 'Celltype', use_raw=True)

In [ ]:
sc.pl.dendrogram(adata, 'Celltype')

In [ ]:
sc.pl.correlation_matrix(adata, 'Celltype')

## Double positive cells

In [ ]:
gene1 = 'ASCL2'
gene2 = 'LGR5'

In [ ]:
print(sum(adata[adata.obs['Condition']=='HEALTHY'].raw[:,'{}'.format(gene1)].X.todense() > 0)[0,0])
print(sum(adata[adata.obs['Condition']=='HEALTHY'].raw[:,'{}'.format(gene2)].X.todense() > 0)[0,0])

print(sum(adata[adata.obs['Condition']=='INFLAMED'].raw[:,'{}'.format(gene1)].X.todense() > 0)[0,0])
print(sum(adata[adata.obs['Condition']=='INFLAMED'].raw[:,'{}'.format(gene2)].X.todense() > 0)[0,0])

In [ ]:
a = sum(adata[adata.obs['Condition']=='HEALTHY'].raw[:,'{}'.format(gene2)].X.todense() > 0)[0,0]

b = sum((adata[adata.obs['Condition']=='HEALTHY'].raw[:,'{}'.format(gene2)].X.todense() > 0) & ~ 
          (adata[adata.obs['Condition']=='HEALTHY'].raw[:,'{}'.format(gene1)].X.todense() > 0))[0,0]

print(a,b,100*b/a)

In [ ]:
a = sum(adata[adata.obs['Condition']=='INFLAMED'].raw[:,'{}'.format(gene2)].X.todense() > 0)[0,0]

b = sum((adata[adata.obs['Condition']=='INFLAMED'].raw[:,'{}'.format(gene2)].X.todense() > 0) & ~ 
          (adata[adata.obs['Condition']=='INFLAMED'].raw[:,'{}'.format(gene1)].X.todense() > 0))[0,0]

print(a,b,100*b/a)

In [ ]:
sc.set_figure_params(fontsize=15, vector_friendly=False, dpi=150, dpi_save=100)

adata.obs['CoEx'] = (adata.raw[:,'{}'.format(gene1)].X.todense() > 0) & (adata.raw[:,'{}'.format(gene2)].X.todense() > 0)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10,5))
grey_black = colors.ListedColormap(['#D3D3D3', '#000000'])

sc.pl.umap(adata[adata.obs['Condition']=='HEALTHY'], cmap=grey_black, color='CoEx', size=15,
           title='HEALHTY', ax=ax1, show=False)
ax1.collections[-1].colorbar.remove()
ax1.set_aspect(aspect=1)
sc.pl.umap(adata[adata.obs['Condition']=='NONINFLAMED'], cmap=grey_black, color='CoEx', size=15,
           title='NONINFLAMED', ax=ax2, show=False)
ax2.collections[-1].colorbar.remove()
ax2.set_aspect(aspect=1)
sc.pl.umap(adata[adata.obs['Condition']=='INFLAMED'], cmap=grey_black, color='CoEx', size=15,
           title='INFLAMED', ax=ax3, show=False)
ax3.collections[-1].colorbar.remove()
ax3.set_aspect(aspect=1)

fig.suptitle('Coexpression of {} and {}'.format(gene1, gene2), y=0.85)
fig.savefig('figures/coexp_{}and{}.pdf'.format(gene1, gene2), bbox_inches='tight')

scv.settings.set_figure_params(dpi=150, vector_friendly=False)

In [ ]:
sc.set_figure_params(fontsize=15, vector_friendly=False, dpi=150, dpi_save=100)

adata.obs['ExEx'] = (adata.raw[:,'{}'.format(gene1)].X.todense() > 0) & ~ (adata.raw[:,'{}'.format(gene2)].X.todense() > 0)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10,5))
grey_black = colors.ListedColormap(['#D3D3D3', '#000000'])

sc.pl.umap(adata[adata.obs['Condition']=='HEALTHY'], cmap=grey_black, color='ExEx', size=15,
           title='HEALHTY', ax=ax1, show=False)
ax1.collections[-1].colorbar.remove()
ax1.set_aspect(aspect=1)
sc.pl.umap(adata[adata.obs['Condition']=='NONINFLAMED'], cmap=grey_black, color='ExEx', size=15,
           title='NONINFLAMED', ax=ax2, show=False)
ax2.collections[-1].colorbar.remove()
ax2.set_aspect(aspect=1)
sc.pl.umap(adata[adata.obs['Condition']=='INFLAMED'], cmap=grey_black, color='ExEx', size=15,
           title='INFLAMED', ax=ax3, show=False)
ax3.collections[-1].colorbar.remove()
ax3.set_aspect(aspect=1)

fig.suptitle('Exclusive expression of {} over {}'.format(gene1, gene2), y=0.85, x=0.55)
fig.savefig('figures/exexp_{}over{}.pdf'.format(gene1, gene2), bbox_inches='tight')

scv.settings.set_figure_params(dpi=150, vector_friendly=False)

# IBD signature 

Liu, J. Z. et al.

In [ ]:
features_orig_Liu = ['USP1', 'BTBD8', 'SLC30A', 'EDG1', 'SELP', 'SELE', 'SELL', 'PTGS2', 'PLA2G4A', 'PTPRC', 'MARCH7', 'LY75', 'PLA2R1', 'ICOS', 'CD28', 'CTLA4', 'CCL20', 'PDCD1', 'ATG4B', 'FLJ78302', 'LTF', 'CCR1', 'CCR2', 'CCR3', 'CCR5', 'NFKBIZ', 'HGFAC', 'OSMR', 'FYB', 'LIFR', 'C5orf4', 'DUSP1', 'IRF4', 'DUSP22', 'MAP3K7IP2', 'AHR', 'CNTNAP2', 'PTK2B', 'TRIM35', 'EPHX2', 'NFKB2', 'TRIM8', 'TMEM180', 'CD27', 'TNFRSF1A', 'LTBR', 'SH2B3', 'ALDH2', 'ATXN2', 'PRKAB1', 'AKAP1', 'TFSF11', 'NFATC1', 'TST', 'TEF', 'NHP2L1', 'PMM1', 'L3MBTL2', 'CHADL']

features_Liu = [feature for feature in features_orig_Liu if feature in adata.var_names]

sc.pl.umap(adata, color=features_Liu, cmap='YlOrRd', save='Liu_IBD_genes')

In [ ]:
sc.tl.score_genes(adata, features_Liu, score_name='IBD_Liu')

In [ ]:
sc.pl.umap(adata, color='IBD_Liu', cmap='bwr')

In [ ]:
# Elk3, Tead4, Fos and Zfp57

sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
gene = 'IBD_Liu'

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10,5))

fig.tight_layout(pad=1.5)

fig.suptitle('{} expression'.format(gene), y=0.85, x=0.55)

sc.pl.umap(adata[adata.obs['Condition']=='HEALTHY'], color=['{}'.format(gene)], title='HEALTHY', ax=ax1, show=False, 
           size=15, cmap='bwr', vmax=max(adata.obs['IBD_Liu']), vmin=min(adata.obs['IBD_Liu']))
ax1.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='NONINFLAMED'], color=['{}'.format(gene)], title='NONINFLAMED', ax=ax2, show=False, 
           size=15, cmap='bwr', vmax=max(adata.obs['IBD_Liu']), vmin=min(adata.obs['IBD_Liu']))
ax2.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='INFLAMED'], color=['{}'.format(gene)], title='INFLAMED', ax=ax3, show=False, 
           size=15, cmap='bwr', vmax=max(adata.obs['IBD_Liu']), vmin=min(adata.obs['IBD_Liu']))
ax3.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

fig.savefig('figures/profile_score_Liu_IBD_genes.pdf', bbox_inches='tight')

de Lange, K. M. et al

In [ ]:
features_orig_Lange = ['SLAMF8', 'ITGA4', 'ITGB8', 'PLCG2']

features_Lange = [feature for feature in features_orig_Lange if feature in adata.var_names]

sc.pl.umap(adata, color=features_Lange, cmap='YlOrRd', save='Lange_IBD_genes')

In [ ]:
sc.set_figure_params(fontsize=10, vector_friendly=True, dpi=150, dpi_save=300)

In [ ]:
sc.pl.dotplot(adata, features_Lange, groupby='Condition', color_map='bwr', dendrogram=False, standard_scale='var', save='Lange_IBD_genes')

In [ ]:
sc.tl.score_genes(adata, features_Lange, score_name='IBD_Lange')

In [ ]:
sc.pl.umap(adata, color='IBD_Lange', cmap='bwr')

In [ ]:
# Elk3, Tead4, Fos and Zfp57

sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
gene = 'IBD_Lange'

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10,5))

fig.tight_layout(pad=1.5)

fig.suptitle('{} expression'.format(gene), y=0.85, x=0.55)

sc.pl.umap(adata[adata.obs['Condition']=='HEALTHY'], color=['{}'.format(gene)], title='HEALTHY', ax=ax1, show=False, 
           size=15, cmap='bwr', vmax=max(adata.obs['IBD_Lange']), vmin=min(adata.obs['IBD_Lange']))
ax1.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='NONINFLAMED'], color=['{}'.format(gene)], title='NONINFLAMED', ax=ax2, show=False, 
           size=15, cmap='bwr', vmax=max(adata.obs['IBD_Lange']), vmin=min(adata.obs['IBD_Lange']))
ax2.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='INFLAMED'], color=['{}'.format(gene)], title='INFLAMED', ax=ax3, show=False, 
           size=15, cmap='bwr', vmax=max(adata.obs['IBD_Lange']), vmin=min(adata.obs['IBD_Lange']))
ax3.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

fig.savefig('figures/profile_score_Lange_IBD_genes.pdf', bbox_inches='tight')

### DE LAnge and extra

In [ ]:
with open('IBD_figures/UC.txt', 'r') as f:
    UC_genes = f.read().splitlines()
    
with open('IBD_figures/CD.txt', 'r') as f:
    CD_genes = f.read().splitlines()
    
with open('IBD_figures/IBD.txt', 'r') as f:
    IBD_genes = f.read().splitlines()

In [ ]:
UC_genes = list(set(UC_genes))
UC_genes = [feature for feature in UC_genes if feature in adata.var_names]

CD_genes = list(set(CD_genes))
CD_genes = [feature for feature in CD_genes if feature in adata.var_names]

IBD_genes = list(set(IBD_genes))
IBD_genes = [feature for feature in IBD_genes if feature in adata.var_names]

In [ ]:
# Input
from scipy.sparse import issparse
groupby = 'Condition'
var_names = UC_genes

# Tidy data
matrix = adata.raw[:, var_names].X
if issparse(matrix):
    matrix = matrix.toarray()
obs_tidy = pd.DataFrame(matrix, columns=var_names)
categorical = adata.obs[groupby]
obs_tidy.set_index(categorical, groupby, inplace=True)
categories = obs_tidy.index.categories

# Mean expression and scaled
mean_obs = obs_tidy.groupby(level=0).mean()
mean_obs -= mean_obs.min(0)
mean_obs = (mean_obs / mean_obs.max(0)).fillna(0)

In [ ]:
mean_obs.transpose().to_csv('UC_genes.csv')

# IBD genes

In [ ]:
with open('IBD_figures/UC.txt', 'r') as f:
    UC_genes = f.read().splitlines()
    
with open('IBD_figures/CD.txt', 'r') as f:
    CD_genes = f.read().splitlines()
    
with open('IBD_figures/IBD.txt', 'r') as f:
    IBD_genes = f.read().splitlines()

In [ ]:
UC_genes = list(set(UC_genes))
print(len(UC_genes))
UC_genes = [feature for feature in UC_genes if feature in adata.var_names]
print(len(UC_genes))

CD_genes = list(set(CD_genes))
print(len(CD_genes))
CD_genes = [feature for feature in CD_genes if feature in adata.var_names]
print(len(CD_genes))

IBD_genes = list(set(IBD_genes))
print(len(IBD_genes))
IBD_genes = [feature for feature in IBD_genes if feature in adata.var_names]
print(len(IBD_genes))

In [ ]:
sc.tl.score_genes(adata, UC_genes, score_name='UC_genes')

In [ ]:
sc.tl.score_genes(adata, CD_genes, score_name='CD_genes')

In [ ]:
sc.tl.score_genes(adata, IBD_genes, score_name='IBD_genes')

In [ ]:
sc.pl.umap(adata, color='IBD_genes', cmap='bwr')

In [ ]:
# Elk3, Tead4, Fos and Zfp57

sc.set_figure_params(fontsize=12, vector_friendly=True, dpi=150, dpi_save=300)
gene = 'IBD_genes'

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10,5))

fig.tight_layout(pad=1.5)

fig.suptitle('{} expression'.format(gene), y=0.85, x=0.55)

sc.pl.umap(adata[adata.obs['Condition']=='HEALTHY'], color=['{}'.format(gene)], title='HEALTHY', ax=ax1, show=False, 
           size=15, cmap='bwr')
ax1.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='NONINFLAMED'], color=['{}'.format(gene)], title='NONINFLAMED', ax=ax2, show=False, 
           size=15, cmap='bwr')
ax2.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

sc.pl.umap(adata[adata.obs['Condition']=='INFLAMED'], color=['{}'.format(gene)], title='INFLAMED', ax=ax3, show=False, 
           size=15, cmap='bwr')
ax3.set_aspect(aspect=1)
fig = plt.gcf()
cbar_ax = fig.axes[-1]
cbar_ax.set_aspect(aspect=10)

#fig.savefig('figures/all_compare_{}.pdf'.format(gene), bbox_inches='tight')

ax=sc.pl.violin(adata, '{}'.format(gene), groupby='Condition', scale='count', palette=['#808080', '#E5CC00', '#8B0000'], 
             rotation=45, size=0.1, show=False, alpha=0.1, inner='box', stripplot=False, cut=0)
ax.grid(False)
ax.set_xticklabels(['HEALTHY', 'NON-ULCERATED', 'ULCERATED'])
#plt.savefig('figures/{}_violin.pdf'.format(gene), bbox_inches = "tight")
#